# PyTorch Script Mode — Batch Transform

SageMaker Script Mode com DLC `pytorch-inference` da AWS.

- **Sem Dockerfile** e **sem push para ECR** — usa a imagem gerenciada pela AWS
- Funções `model_fn / input_fn / predict_fn / output_fn` em `app/pytorch/inference.py`
- Modelo no formato `state dict (`.pth`) empacotado junto com o código em `code/`
- `BatchStrategy: SingleRecord` — cada linha do JSONL vira um request independente
- Formato de entrada simples: `{"record_id": "...", "image_s3_path": "s3://..."}`

```
model_pytorch_script.tar.gz
├── ResNet18.pth
└── code/
    ├── inference.py
    ├── s3_utils.py
    └── requirements.txt
```

## Local (teste direto das funções de inferência)

No Script Mode não há container próprio para rodar localmente.
O teste local consiste em chamar as 4 funções na mesma ordem que o DLC faz internamente.

In [1]:
import sys
import json
sys.path.insert(0, '..')

### model_fn — carrega o modelo (executado uma vez na inicialização)

In [2]:
from app.pytorch.inference import model_fn, input_fn, predict_fn, output_fn

# Equivalente a /opt/ml/model/ no container
model_ctx = model_fn('../models')

print(f"Device: {model_ctx['device']}")
print(f"Modelo: {type(model_ctx['model']).__name__}")

ModuleNotFoundError: No module named 's3_utils'

### input_fn / predict_fn / output_fn — ciclo completo de inferência

In [ ]:
# Simula o body que o SageMaker envia a cada POST /invocations (SingleRecord)
request_body = json.dumps({
    "record_id":     "cardboard1",
    "image_s3_path": "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/cardboard1.jpg",
})

data             = input_fn(request_body, "application/json")
result           = predict_fn(data, model_ctx)
body, ctype      = output_fn(result, "application/jsonlines")

print(f"Content-Type: {ctype}")
print(f"Resposta:     {body}")

In [ ]:
# Batch local: múltiplos registros numa única chamada (simula MultiRecord ou chamada direta)
batch_body = "\n".join(json.dumps({
    "record_id":     rid,
    "image_s3_path": f"s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/{rid}.jpg",
}) for rid in ["cardboard1", "metal1", "paper1", "plastic1", "trash1"])

data        = input_fn(batch_body, "application/jsonlines")
result      = predict_fn(data, model_ctx)
body, _     = output_fn(result, "application/jsonlines")

for line in body.splitlines():
    print(line)

## AWS — Batch Transform

In [29]:
import boto3
import json
import random
import subprocess
import time
import pandas as pd
import sagemaker
import sagemaker.core.image_uris
from datetime import datetime
from urllib.parse import urlparse

In [30]:
session    = boto3.session.Session(region_name="us-east-1")
region     = session.region_name or "us-east-1"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Região:     {region}")
print(f"Account ID: {account_id}")

Região:     us-east-1
Account ID: 891377318910


In [31]:
configs = sagemaker.core.image_uris.config_for_framework("pytorch")
configs.keys()

dict_keys(['eia', 'inference', 'inference_graviton', 'training'])

In [32]:
print(configs['inference'].keys())
print(f"Processors: {configs['inference']['processors']}")

dict_keys(['processors', 'version_aliases', 'versions'])
Processors: ['cpu', 'gpu']


In [33]:
# configs['inference']['version_aliases']

In [34]:
SAGEMAKER_ROLE_ARN = f"arn:aws:iam::{account_id}:role/SageMakerExecutionRole-DSA-P3"

S3_BUCKET      = "mlops-us-east-1-891377318910"
S3_MODEL_KEY   = "sagemaker-batch-transform/refinamento/model/model_pytorch_script.tar.gz"
MODEL_DATA_URL = f"s3://{S3_BUCKET}/{S3_MODEL_KEY}"

INPUT_S3  = f"s3://{S3_BUCKET}/sagemaker-batch-transform/refinamento/input/"
OUTPUT_S3 = f"s3://{S3_BUCKET}/sagemaker-batch-transform/refinamento/output/output-pytorch-script/"

# URI da DLC pytorch-inference gerenciada pela AWS (sem necessidade de ECR próprio)
DLC_IMAGE_URI = sagemaker.core.image_uris.retrieve(
    framework     = "pytorch",
    region        = region,
    version       = "2.5.1",
    py_version    = "py311",
    image_scope   = "inference",
    instance_type = "ml.g4dn.xlarge",
)

print(f"Role ARN:       {SAGEMAKER_ROLE_ARN}")
print(f"Model data URL: {MODEL_DATA_URL}")
print(f"DLC Image URI:  {DLC_IMAGE_URI}")
print(f"Input S3:       {INPUT_S3}")
print(f"Output S3:      {OUTPUT_S3}")

Role ARN:       arn:aws:iam::891377318910:role/SageMakerExecutionRole-DSA-P3
Model data URL: s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/model/model_pytorch_script.tar.gz
DLC Image URI:  763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.5.1-gpu-py311
Input S3:       s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/
Output S3:      s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/output/output-pytorch-script/


### Empacotar Modelo + Código

O script `package_pytorch_script.py` cria o `model_pytorch_script.tar.gz` com a estrutura
esperada pelo SageMaker Script Mode:
```
ResNet18.pth
code/inference.py
code/s3_utils.py
code/requirements.txt
```

In [19]:
result = subprocess.run(
    [
        "python", "scripts/package_pytorch_script.py",
        "--model",  "models/ResNet18.pth",
        "--output", "model_sagemaker/model_pytorch_script.tar.gz"
    ],
    capture_output=True,
    text=True,
    cwd="..",
)

print(result.stdout)
if result.returncode != 0:
    print("ERRO:", result.stderr)

Gerado: model_sagemaker/model_pytorch_script.tar.gz  (42.3 MB)

Conteúdo do arquivo:
  ResNet18.pth
  code/inference.py
  code/s3_utils.py
  code/requirements.txt

Próximo passo:
  aws s3 cp model_sagemaker/model_pytorch_script.tar.gz s3://<bucket>/<prefix>/model_pytorch_script.tar.gz



### Upload para S3

In [35]:
S3_MODEL_KEY

'sagemaker-batch-transform/refinamento/model/model_pytorch_script.tar.gz'

In [36]:
print(f"Modelo enviado para: {MODEL_DATA_URL}")

boto3.client("s3").upload_file(
    "../model_sagemaker/model_pytorch_script.tar.gz",
    S3_BUCKET,
    S3_MODEL_KEY,
)

Modelo enviado para: s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/model/model_pytorch_script.tar.gz


### Criar Input Metadata File

Formato simples — cada linha contém apenas `record_id` e `image_s3_path`.  
Diferente do Triton (KServe v2), o Script Mode recebe JSON puro sem envelope de tensores.

In [37]:
_parsed = urlparse(INPUT_S3)
_bucket = _parsed.netloc
_prefix = _parsed.path.lstrip("/")

_pag = boto3.client("s3").get_paginator("list_objects_v2")
all_image_keys = [
    obj["Key"]
    for page in _pag.paginate(Bucket=_bucket, Prefix=_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Imagens disponíveis: {len(all_image_keys)}")

Imagens disponíveis: 45


In [38]:
qt_paylaods = 5  # len(all_image_keys)
_sample = random.sample(all_image_keys, qt_paylaods)

_lines = [
    json.dumps({
        "record_id":     f"r{i:04d}",
        "image_s3_path": f"s3://{_bucket}/{key}",
    })
    for i, key in enumerate(_sample)
]

metadata_key = "sagemaker-batch-transform/refinamento/metadata/metadata_pytorch_script.jsonl"
boto3.client("s3").put_object(
    Bucket=_bucket,
    Key=metadata_key,
    Body="\n".join(_lines).encode("utf-8"),
    ContentType="application/jsonlines",
)

metadata_s3_uri = f"s3://{_bucket}/{metadata_key}"
print(f"Metadata S3 URI: {metadata_s3_uri}")
print(f"\nExemplo da 1ª linha:\n{_lines[0]}")

Metadata S3 URI: s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/metadata/metadata_pytorch_script.jsonl

Exemplo da 1ª linha:
{"record_id": "r0000", "image_s3_path": "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/plastic9.jpg"}


### Criar SageMaker Model

No Script Mode, `Image` aponta para a DLC da AWS (sem ECR próprio).  
`SAGEMAKER_PROGRAM` indica ao DLC qual script carregar — coincide com `code/inference.py` dentro do tar.gz.

In [39]:
sm_client = boto3.client("sagemaker", region_name=region)

In [40]:
timestamp  = datetime.now().strftime("%Y%m%d%H%M%S")
model_name = f"resnet18-pytorch-script-{timestamp}"
print(f"Nome do model: {model_name}")

Nome do model: resnet18-pytorch-script-20260627130236


In [41]:
response_model = sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image":        DLC_IMAGE_URI,
        "ModelDataUrl": MODEL_DATA_URL,
        "Environment": {
            # Diz ao DLC qual script carregar dentro de code/
            "SAGEMAKER_PROGRAM": "inference.py",
        },
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN,
)

model_arn = response_model["ModelArn"]
print(f"Model ARN: {model_arn}")

Model ARN: arn:aws:sagemaker:us-east-1:891377318910:model/resnet18-pytorch-script-20260627130236


In [ ]:
# Delete Sagemaker Model
resp = sm_client.delete_model(ModelName=model_name)
resp

### Criar Batch Transform Job

`SingleRecord` + `MaxConcurrentTransforms=8`: o SageMaker envia até 8 requests simultâneos
ao container — cada um com 1 linha JSON. O DLC chama as funções em paralelo (threads separadas).

In [42]:
job_name       = f"batch-pytorch-script-{timestamp}"
instance_type  = "ml.g4dn.xlarge"
instance_count = 1

response_job = sm_client.create_transform_job(
    TransformJobName=job_name,
    ModelName=model_name,
    BatchStrategy="SingleRecord",
    MaxConcurrentTransforms=8,
    MaxPayloadInMB=1,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri":      metadata_s3_uri,
            }
        },
        "ContentType": "application/json",
        "SplitType":   "Line",
    },
    TransformOutput={
        "S3OutputPath": OUTPUT_S3,
        "AssembleWith": "Line",
    },
    TransformResources={
        "InstanceType":  instance_type,
        "InstanceCount": instance_count,
    },
)

job_arn = response_job["TransformJobArn"]
print(f"Nome do job: {job_name}")
print(f"Job ARN:     {job_arn}")

Nome do job: batch-pytorch-script-20260627130236
Job ARN:     arn:aws:sagemaker:us-east-1:891377318910:transform-job/batch-pytorch-script-20260627130236


### Monitorar Status do Job

In [28]:
while True:
    desc   = sm_client.describe_transform_job(TransformJobName=job_name)
    status = desc["TransformJobStatus"]
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {status}")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)

if desc.get("FailureReason"):
    print(f"Falha: {desc['FailureReason']}")
else:
    print(f"Início: {desc.get('TransformStartTime')}")
    print(f"Fim:    {desc.get('TransformEndTime')}")

[12:34:50] InProgress
[12:35:20] InProgress
[12:35:50] InProgress
[12:36:20] InProgress
[12:36:51] InProgress
[12:37:21] InProgress
[12:37:51] InProgress
[12:38:21] InProgress
[12:38:51] InProgress
[12:39:22] InProgress
[12:39:52] InProgress
[12:40:22] InProgress
[12:40:52] InProgress
[12:41:23] InProgress
[12:41:53] InProgress
[12:42:23] InProgress
[12:42:53] InProgress


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:7                                                                                    │
│                                                                                                  │
│    4 │   print(f"[{datetime.now().strftime('%H:%M:%S')}] {status}")                              │
│    5 │   if status in ("Completed", "Failed", "Stopped"):                                        │
│    6 │   │   break                                                                               │
│ ❱  7 │   time.sleep(30)                                                                          │
│    8                                                                                             │
│    9 if desc.get("FailureReason"):                                                               │
│   10 │   print(f"Falha: {desc['FailureReason']}")                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyboardInterrupt

### Ler Resultados do S3

Saída em JSON puro — sem envelope KServe v2.  
Cada linha: `{"record_id": "...", "class_name": "...", "score": 0.999}`

In [ ]:
_parsed_out = urlparse(OUTPUT_S3)
_out_bucket = _parsed_out.netloc
_out_prefix = _parsed_out.path.lstrip("/")

_s3  = boto3.client("s3")
_pag = _s3.get_paginator("list_objects_v2")

output_keys = [
    obj["Key"]
    for page in _pag.paginate(Bucket=_out_bucket, Prefix=_out_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith((".jsonl", ".out"))
]

print(f"Arquivos de saída: {len(output_keys)}")
for k in output_keys:
    print(f"  s3://{_out_bucket}/{k}")

In [ ]:
records = []
for key in output_keys:
    body = _s3.get_object(Bucket=_out_bucket, Key=key)["Body"].read().decode()
    for line in body.strip().splitlines():
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)[["record_id", "class_name", "score"]]
df["score"] = df["score"].astype(float)
df = df.sort_values("record_id").reset_index(drop=True)

print(f"Total de registros: {len(df)}")

In [ ]:
df